In [1]:
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import RFE, chi2, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. โหลดข้อมูล
path = kagglehub.dataset_download("sakshigoyal7/credit-card-customers")
data_raw = pd.read_csv(path + "/BankChurners.csv")

# 2. คัดกรองคอลัมน์ที่ไม่เกี่ยวข้องและ Benchmark Columns ออกทันที
cols_to_drop = [
    "CLIENTNUM",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2",
]
data = data_raw.drop(columns=cols_to_drop)

# 3. แปลง Target Label (Attrition_Flag) และตัวแปรเพศ (Gender)
flag_map = {"Existing Customer": 0, "Attrited Customer": 1}
data["Attrition_Flag"] = data["Attrition_Flag"].map(flag_map)
data["Gender"] = data["Gender"].map({"M": 0, "F": 1})

# 4. กำหนด Order สำหรับ Ordinal Categories (คงค่า Unknown ไว้เป็น NaN ก่อน)
income_order = [
    "Less than $40K",
    "$40K - $60K",
    "$60K - $80K",
    "$80K - $120K",
    "$120K +",
]
data["Income_Category"] = pd.Categorical(
    data["Income_Category"], categories=income_order, ordered=True
).codes
data["Income_Category"] = data["Income_Category"].replace(-1, np.nan)

education_order = [
    "Uneducated",
    "High School",
    "College",
    "Graduate",
    "Post-Graduate",
    "Doctorate",
]
data["Education_Level"] = pd.Categorical(
    data["Education_Level"], categories=education_order, ordered=True
).codes
data["Education_Level"] = data["Education_Level"].replace(-1, np.nan)

data["Marital_Status"] = data["Marital_Status"].replace("Unknown", np.nan)

# =========================================================================
# จุดสำคัญ: แยก Train / Test ก่อนทำ Imputation และ Encoding (ป้องกัน Data Leakage)
# =========================================================================
y = data["Attrition_Flag"]
X = data.drop(columns=["Attrition_Flag"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. สร้าง Missing Indicator ให้ตรงกับโครงสร้างเดิม
for col in ["Education_Level", "Income_Category", "Marital_Status"]:
    indicator_name = col.split("_")[0] + "_Unknown"
    X_train[indicator_name] = X_train[col].isna().astype(int)
    X_test[indicator_name] = X_test[col].isna().astype(int)

# 6. หาค่า Median / Mode จาก X_train เท่านั้น แล้วนำไป Fill ให้ทั้งสองชุด
med_edu = X_train["Education_Level"].median()
med_inc = X_train["Income_Category"].median()
mode_mar = X_train["Marital_Status"].mode()[0]

X_train["Education_Level"] = X_train["Education_Level"].fillna(med_edu)
X_test["Education_Level"] = X_test["Education_Level"].fillna(med_edu)

X_train["Income_Category"] = X_train["Income_Category"].fillna(med_inc)
X_test["Income_Category"] = X_test["Income_Category"].fillna(med_inc)

X_train["Marital_Status"] = X_train["Marital_Status"].fillna(mode_mar)
X_test["Marital_Status"] = X_test["Marital_Status"].fillna(mode_mar)

# 7. One-Hot Encoding สำหรับ Nominal Categorical ('Card_Category', 'Marital_Status')
ohe_cols = ["Card_Category", "Marital_Status"]
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Fit บน X_train และ Transform ทั้งสองชุด
train_ohe = ohe.fit_transform(X_train[ohe_cols])
test_ohe = ohe.transform(X_test[ohe_cols])
ohe_feature_names = ohe.get_feature_names_out(ohe_cols)

train_ohe_df = pd.DataFrame(
    train_ohe, columns=ohe_feature_names, index=X_train.index
)
test_ohe_df = pd.DataFrame(
    test_ohe, columns=ohe_feature_names, index=X_test.index
)

# รวมตารางและนำคอลัมน์ตั้งต้นของ OHE ออก
X_train = pd.concat([X_train.drop(columns=ohe_cols), train_ohe_df], axis=1)
X_test = pd.concat([X_test.drop(columns=ohe_cols), test_ohe_df], axis=1)

# ปรับชนิดข้อมูลที่เป็น Boolean/Dummy ให้เป็น int/float ป้องกันข้อผิดพลาดในการคำนวณ
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# รวมกลับเป็น DataFrame กลางสำหรับฟังก์ชันหรือการแสดงผลที่ต้องการ
data = pd.concat([y_train, X_train], axis=1)

# 8. Feature Scaling (Fit จาก X_train เท่านั้น)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 9. Feature Selection (ANOVA F-test, Chi-square, Mutual Info, RFE)
categorical_features = [
    c
    for c in X_train.columns
    if "Unknown" in c
    or "Card_Category" in c
    or "Marital" in c
    or c in ["Gender", "Education_Level", "Income_Category"]
]
numerical_features = [
    c for c in X_train.columns if c not in categorical_features
]

# ANOVA + MI สำหรับ Numerical
f_scores, f_pvalues = f_classif(X_train[numerical_features], y_train)
mi_scores_num = mutual_info_classif(
    X_train[numerical_features], y_train, random_state=42
)

numerical_df = pd.DataFrame(
    {
        "feature": numerical_features,
        "anova_f": f_scores,
        "anova_pvalue": f_pvalues,
        "mutual_info": mi_scores_num,
    }
).sort_values("anova_f", ascending=False)

# Chi-Square + MI สำหรับ Categorical
chi_scores, chi_pvalues = chi2(X_train[categorical_features], y_train)
mi_scores_cat = mutual_info_classif(
    X_train[categorical_features],
    y_train,
    discrete_features=True,
    random_state=42,
)

categorical_df = pd.DataFrame(
    {
        "feature": categorical_features,
        "chi2_score": chi_scores,
        "chi2_pvalue": chi_pvalues,
        "mutual_info": mi_scores_cat,
    }
).sort_values("chi2_score", ascending=False)

# รวมตารางกรองผล
filter_df = pd.concat(
    [
        numerical_df.rename(
            columns={"anova_f": "score", "anova_pvalue": "pvalue"}
        ),
        categorical_df.rename(
            columns={"chi2_score": "score", "chi2_pvalue": "pvalue"}
        ),
    ],
    ignore_index=True,
)

# RFE ด้วย Logistic Regression
log_reg = LogisticRegression(
    max_iter=2000, random_state=42, class_weight="balanced"
)
rfe = RFE(estimator=log_reg, n_features_to_select=10)
rfe.fit(X_scaled, y_train)

wrapper_df = pd.DataFrame(
    {
        "feature": X_train.columns,
        "rfe_selected": rfe.support_,
        "rfe_rank": rfe.ranking_,
    }
).sort_values("rfe_rank")

compare = filter_df.merge(wrapper_df, on="feature")

# โหวตคัดเลือกฟีเจอร์เด่น
compare["vote_significant"] = (compare["pvalue"] < 0.05).astype(int)
compare["rank_mi"] = compare["mutual_info"].rank(ascending=False)
compare["vote_mi"] = (compare["rank_mi"] <= 10).astype(int)
compare["vote_rfe"] = compare["rfe_selected"].astype(int)
compare["votes"] = (
    compare["vote_significant"] + compare["vote_mi"] + compare["vote_rfe"]
)

final_ranking = compare.sort_values("votes", ascending=False)
selected_features = final_ranking[final_ranking["votes"] >= 3][
    "feature"
].tolist()

print(f"ขนาดชุดข้อมูล: Train {X_train.shape}, Test {X_test.shape}")
print(f"ฟีเจอร์ที่ผ่านเกณฑ์โหวต (>=3): {selected_features}")

c:\Users\Natti\anaconda3\envs\ml_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ขนาดชุดข้อมูล: Train (8101, 27), Test (2026, 27)
ฟีเจอร์ที่ผ่านเกณฑ์โหวต (>=3): ['Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Total_Revolving_Bal', 'Contacts_Count_12_mon', 'Total_Trans_Amt', 'Months_Inactive_12_mon', 'Total_Relationship_Count']
